In [1]:
import pandas as pd

In [2]:
class MissingDict(dict):
    __missing__ = lambda self, key: key

map_values = {"Brighton and Hove Albion": "Brighton", "Huddersfield Town":"Huddersfield","Manchester United": "Manchester Utd", "Newcastle United": "Newcastle Utd", "Nottingham Forest":"Nott'ham Forest","Tottenham Hotspur": "Tottenham","Sheffield United":"Sheffield Utd", "West Ham United": "West Ham","West Bromwich Albion":"West Brom", "Wolverhampton Wanderers": "Wolves","Wycombe Wanderers":"Wycombe","Sheffield Wednesday":"Sheffield Weds","Queens Park Rangers":"QPR","Rotherham United":"Rotherham Utd","Preston North End":"Preston","Peterborough United":"P'borough Utd","Charlton Athletic":"Charlton Ath","Bolton Wanderers":"Bolton","Blackburn Rovers":"Blackburn"} 
mapping = MissingDict(**map_values)

In [3]:
match_df = pd.read_excel(r"C:\Users\George\Documents\University\Masters\DISS\Datasets\Match Dataset with Weights and Previous Season Data (1).xlsx", index_col=0)

In [4]:
avg_pos_df = pd.read_excel(r"C:\Users\George\Documents\University\Masters\DISS\Datasets\Average Position Data (With 2024).xlsx", index_col=0)

In [5]:
avg_pos_df["new_team"] = avg_pos_df['Squad'].map(mapping)
avg_pos_df.drop(columns=["Squad","Pos"], inplace=True)

In [6]:
avg_pos_df.reset_index(drop=True, inplace=True)

In [7]:
avg_pos_df

,Season,Avg Pos Last 3,new_team
0,2019,61.666667,AFC Wimbledon
1,2020,63.333333,AFC Wimbledon
2,2021,63.666667,AFC Wimbledon
3,2024,64.666667,AFC Wimbledon
4,2021,58.000000,Acc'ton Stanley
...,...,...,...
439,2024,12.333333,Wolves
440,2021,50.000000,Wycombe
441,2022,46.333333,Wycombe
442,2023,48.333333,Wycombe


In [8]:
avg_pos_df2018 = avg_pos_df[avg_pos_df["Season"] == 2018]
avg_pos_df2019 = avg_pos_df[avg_pos_df["Season"] == 2019]
avg_pos_df2020 = avg_pos_df[avg_pos_df["Season"] == 2020]
avg_pos_df2021 = avg_pos_df[avg_pos_df["Season"] == 2021]
avg_pos_df2022 = avg_pos_df[avg_pos_df["Season"] == 2022]
avg_pos_df2023 = avg_pos_df[avg_pos_df["Season"] == 2023]
avg_pos_df2024 = avg_pos_df[avg_pos_df["Season"] == 2024]

In [9]:
def add_avg_pos(df, season, avg_pos_df):
    for season, pos, team in avg_pos_df.values:
        df = df.loc[df['season'] == season]
        #df.loc[df['opponent'] == team, 'opponent_avg_pos'] = pos
        df.loc[df['new_team'] == team, 'new_team_avg_pos'] = pos  
    return df

In [10]:
df2017 = match_df[match_df["season"] == 2017]
df2018 = add_avg_pos(match_df, 2018, avg_pos_df2018)
df2019 = add_avg_pos(match_df, 2019, avg_pos_df2019)
df2020 = add_avg_pos(match_df, 2020, avg_pos_df2020)
df2021 = add_avg_pos(match_df, 2021, avg_pos_df2021)
df2022 = add_avg_pos(match_df, 2022, avg_pos_df2022)
df2023 = add_avg_pos(match_df, 2023, avg_pos_df2023)
df2024 = add_avg_pos(match_df, 2024, avg_pos_df2024)

C:\Users\dmsta\AppData\Local\Temp\ipykernel_2688\3456583746.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df['new_team'] == team, 'new_team_avg_pos'] = pos
C:\Users\dmsta\AppData\Local\Temp\ipykernel_2688\3456583746.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df['new_team'] == team, 'new_team_avg_pos'] = pos


C:\Users\dmsta\AppData\Local\Temp\ipykernel_2688\3456583746.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df['new_team'] == team, 'new_team_avg_pos'] = pos
C:\Users\dmsta\AppData\Local\Temp\ipykernel_2688\3456583746.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df['new_team'] == team, 'new_team_avg_pos'] = pos
C:\Users\dmsta\AppData\Local\Temp\ipykernel_2688\3456583746.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

In [11]:
df = pd.concat([df2017, df2018, df2019, df2020, df2021, df2022, df2023, df2024])

In [12]:
df.reset_index(drop=True, inplace=True)

In [13]:
def rolling_averages(group, number, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(number, closed='left').mean()
    group[new_cols] = rolling_stats
    #group = group.dropna(subset=new_cols)
    return group

In [14]:
# Setting Champ games all to 1 point each and xGD to -0.5 and GD to -0.5
constant_points = []
constant_xGD = []
constant_GD = []
for x in range(len(df)):
    if df["comp"][x] == "Championship":
        constant_points.append(1)
        constant_xGD.append(-1)
        constant_GD.append(-1)
    else:
        constant_points.append(df["league_weighted_points"][x])
        constant_xGD.append(df["league_weighted_xGD"][x])
        constant_GD.append(df["league_weighted_GD"][x])

df["constant_points"] = constant_points
df["constant_xGD"] = constant_xGD
df["constant_GD"] = constant_GD


In [15]:
pd.set_option('display.max_columns', None)
df

,date,comp,round,venue,result,gf,ga,opponent,xg,xga,season,team,new_team,xGD,GD,points,league_diff,xGD_league_diff,GD_league_diff,GF_league_diff,GA_league_diff,xG_league_diff,xGA_league_diff,league_weighted_points,league_weighted_xGD,league_weighted_GD,league_weighted_GF,league_weighted_GA,league_weighted_xG,league_weighted_xGA,new_team_previous_points,new_team_previous_xGD,new_team_previous_GD,new_team_previous_position,new_team_previous_expected_position,Adjusted GS,Adjusted GC,Adjusted GD,new_team_avg_pos,constant_points,constant_xGD,constant_GD
0,2017-08-05,Championship,Matchweek 1,Home,W,1.0,0.0,Middlesbrough,NaN,NaN,2017,Wolverhampton Wanderers,Wolves,NaN,1.0,3.0,0.3,-1.25,-1.78,-0.76,1.02,-0.76,1.02,0.9,NaN,-0.78,0.24,1.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0
1,2017-08-12,Championship,Matchweek 2,Away,W,2.0,0.0,Derby County,NaN,NaN,2017,Wolverhampton Wanderers,Wolves,NaN,2.0,3.0,0.3,-1.25,-1.78,-0.76,1.02,-0.76,1.02,0.9,NaN,0.22,1.24,1.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0
2,2017-08-15,Championship,Matchweek 3,Away,W,3.0,2.0,Hull City,NaN,NaN,2017,Wolverhampton Wanderers,Wolves,NaN,1.0,3.0,0.3,-1.25,-1.78,-0.76,1.02,-0.76,1.02,0.9,NaN,-0.78,2.24,3.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0
3,2017-08-19,Championship,Matchweek 4,Home,L,1.0,2.0,Cardiff City,NaN,NaN,2017,Wolverhampton Wanderers,Wolves,NaN,-1.0,0.0,0.3,-1.25,-1.78,-0.76,1.02,-0.76,1.02,0.0,NaN,-2.78,0.24,3.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0
4,2017-08-26,Championship,Matchweek 5,Away,D,0.0,0.0,Brentford,NaN,NaN,2017,Wolverhampton Wanderers,Wolves,NaN,0.0,1.0,0.3,-1.25,-1.78,-0.76,1.02,-0.76,1.02,0.3,NaN,-1.78,-0.76,1.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13803,2025-04-08,Premier League,Matchweek 34,Home,NaN,NaN,NaN,Leicester City,NaN,NaN,2024,Wolves,Wolves,NaN,NaN,NaN,1.0,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,-20.7,-15.0,15.0,17.0,NaN,NaN,NaN,12.333333,NaN,NaN,NaN
13804,2025-04-15,Premier League,Matchweek 35,Away,NaN,NaN,NaN,Manchester City,NaN,NaN,2024,Wolves,Wolves,NaN,NaN,NaN,1.0,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,-20.7,-15.0,15.0,17.0,NaN,NaN,NaN,12.333333,NaN,NaN,NaN
13805,2025-04-22,Premier League,Matchweek 36,Home,NaN,NaN,NaN,Brighton,NaN,NaN,2024,Wolves,Wolves,NaN,NaN,NaN,1.0,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,-20.7,-15.0,15.0,17.0,NaN,NaN,NaN,12.333333,NaN,NaN,NaN
13806,2025-04-29,Premier League,Matchweek 37,Away,NaN,NaN,NaN,Crystal Palace,NaN,NaN,2024,Wolves,Wolves,NaN,NaN,NaN,1.0,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,-20.7,-15.0,15.0,17.0,NaN,NaN,NaN,12.333333,NaN,NaN,NaN


In [16]:
numbers = [5]
for number in numbers:
    cols = ["league_weighted_points","league_weighted_xGD","league_weighted_GD",
            "constant_points","constant_xGD","constant_GD","Adjusted GD"]
    new_cols = [f"{col} form last {number}" for col in cols]
    df = df.groupby("new_team").apply(rolling_averages, number, cols, new_cols)
    df = df.droplevel('new_team')

In [17]:
df[df['season'] == 2023]

,date,comp,round,venue,result,gf,ga,opponent,xg,xga,season,team,new_team,xGD,GD,points,league_diff,xGD_league_diff,GD_league_diff,GF_league_diff,GA_league_diff,xG_league_diff,xGA_league_diff,league_weighted_points,league_weighted_xGD,league_weighted_GD,league_weighted_GF,league_weighted_GA,league_weighted_xG,league_weighted_xGA,new_team_previous_points,new_team_previous_xGD,new_team_previous_GD,new_team_previous_position,new_team_previous_expected_position,Adjusted GS,Adjusted GC,Adjusted GD,new_team_avg_pos,constant_points,constant_xGD,constant_GD,league_weighted_points form last 5,league_weighted_xGD form last 5,league_weighted_GD form last 5,constant_points form last 5,constant_xGD form last 5,constant_GD form last 5,Adjusted GD form last 5
12326,2023-08-12,Premier League,Matchweek 1,Home,W,2.0,1.0,Nott'ham Forest,0.8,1.2,2023,Arsenal,Arsenal,-0.4,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,-0.4,1.0,2.0,1.0,0.8,1.2,84.0,29.3,45.0,2.0,3.0,1.40,1.10,0.30,5.0,3.0,-0.4,1.0,1.8,0.48,1.0,1.8,0.48,1.0,0.74
12327,2023-08-21,Premier League,Matchweek 2,Away,W,1.0,0.0,Crystal Palace,2.0,1.0,2023,Arsenal,Arsenal,1.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,1.0,1.0,0.0,2.0,1.0,84.0,29.3,45.0,2.0,3.0,1.50,0.50,1.00,5.0,3.0,1.0,1.0,1.8,0.22,0.8,1.8,0.22,0.8,0.51
12328,2023-08-26,Premier League,Matchweek 3,Home,D,2.0,2.0,Fulham,3.2,0.6,2023,Arsenal,Arsenal,2.6,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.6,0.0,2.0,2.0,3.2,0.6,84.0,29.3,45.0,2.0,3.0,2.60,1.30,1.30,5.0,1.0,2.6,0.0,1.8,0.42,0.6,1.8,0.42,0.6,0.51
12329,2023-09-03,Premier League,Matchweek 4,Home,W,3.0,1.0,Manchester Utd,2.3,0.9,2023,Arsenal,Arsenal,1.4,2.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.4,2.0,3.0,1.0,2.3,0.9,84.0,29.3,45.0,2.0,3.0,2.65,0.95,1.70,5.0,3.0,1.4,2.0,2.0,1.10,1.2,2.0,1.10,1.2,1.15
12330,2023-09-17,Premier League,Matchweek 5,Away,W,1.0,0.0,Everton,1.0,0.3,2023,Arsenal,Arsenal,0.7,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.7,1.0,1.0,0.0,1.0,0.3,84.0,29.3,45.0,2.0,3.0,1.00,0.15,0.85,5.0,3.0,0.7,1.0,2.6,1.38,1.8,2.6,1.38,1.8,1.59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12815,2024-04-24,Premier League,Matchweek 29,Home,L,0.0,1.0,Bournemouth,0.5,1.9,2023,Wolverhampton Wanderers,Wolves,-1.4,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.4,-1.0,0.0,1.0,0.5,1.9,41.0,-23.3,-27.0,13.0,18.0,0.25,1.45,-1.20,12.0,0.0,-1.4,-1.0,0.4,-0.30,-1.0,0.4,-0.30,-1.0,-0.65
12816,2024-04-27,Premier League,Matchweek 35,Home,W,2.0,1.0,Luton Town,1.2,0.6,2023,Wolverhampton Wanderers,Wolves,0.6,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.6,1.0,2.0,1.0,1.2,0.6,41.0,-23.3,-27.0,13.0,18.0,1.60,0.80,0.80,12.0,3.0,0.6,1.0,0.4,-0.68,-0.8,0.4,-0.68,-0.8,-0.74
12817,2024-05-04,Premier League,Matchweek 36,Away,L,1.0,5.0,Manchester City,0.3,3.7,2023,Wolverhampton Wanderers,Wolves,-3.4,-4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.4,-4.0,1.0,5.0,0.3,3.7,41.0,-23.3,-27.0,13.0,18.0,0.65,4.35,-3.70,12.0,0.0,-3.4,-4.0,0.8,-0.48,-0.6,0.8,-0.48,-0.6,-0.54
12818,2024-05-11,Premier League,Matchweek 37,Home,L,1.0,3.0,Crystal Palace,1.6,2.6,2023,Wolverhampton Wanderers,Wolves,-1.0,-2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-2.0,1.0,3.0,1.6,2.6,41.0,-23.3,-27.0,13.0,18.0,1.30,2.80,-1.50,12.0,0.0,-1.0,-2.0,0.8,-1.22,-1.2,0.8,-1.22,-1.2,-1.21


In [18]:
df.reset_index(drop=True, inplace=True)
df.to_excel(r"C:\Users\George\Documents\University\Masters\DISS\Datasets\Match Dataset with Average Position and General Form (2).xlsx")